In [1]:
import torch
import torch.nn as nn
from d2l import torch as d2l

In [2]:
import math
import os
import random
import collections
from ptbData import load_data_ptb

In [ ]:
batch_size=512
data_iter,vocab = load_data_ptb(batch_size,5,5)

In [26]:


def skip_gram(center,context_negative,embed_v,embed_u):
    v=embed_v(center)
    u=embed_u(context_negative)

    return torch.bmm(v,u.permute(0,2,1)).squeeze(dim=1)

def sigmoidBSE(pred,mask,label):
    return nn.functional.binary_cross_entropy_with_logits(pred,target=label,weight=mask,reduction='none').mean(dim=1)*mask.shape[1]/mask.sum(dim=1)

embed_size=100
embeds=nn.Sequential(nn.Embedding(len(vocab),embed_size),nn.Embedding(len(vocab),embed_size))

def train(net,lr,data_iter,epochs,device):
    def init_weight(m):
        if type(m) == nn.Embedding:
            nn.init.xavier_uniform_(m.weight)

    net.apply(init_weight)
    net=net.to(device)
    updater=torch.optim.Adam(net.parameters(),lr=lr)
    for epoch in range(0,epochs):
        for center,context_negative,mask,label in data_iter:
            updater.zero_grad()
            l=sigmoidBSE(skip_gram(center.to(device),context_negative.to(device),net[0],net[1]),mask.to(device),label.to(device))
            l.sum().backward()
            with torch.no_grad():
                updater.step()
        print(f"epoch {epoch} loss:{l.mean()}")

In [37]:
train(embeds,0.002,data_iter,10,torch.device('cuda:2'))

epoch 0 loss:0.38313889503479004
epoch 1 loss:0.3546832203865051
epoch 2 loss:0.3346177339553833
epoch 3 loss:0.30077826976776123
epoch 4 loss:0.282869428396225
epoch 5 loss:0.2769162654876709
epoch 6 loss:0.26971766352653503
epoch 7 loss:0.2659246325492859
epoch 8 loss:0.2598358690738678
epoch 9 loss:0.26030898094177246


In [38]:
import numpy

def get_similiar_word(query_token,k,embed,vocab):
    idx=vocab[query_token]
    embeds=embed.weight.data
    q_embed=embeds[idx]
    W=torch.mv(embeds,q_embed)/torch.sqrt(torch.sum(embeds*embeds,dim=1)*torch.sum(q_embed*q_embed)+1e-9)
    top_k=torch.topk(W,k=k+1)[1].cpu().numpy().astype('int32')
    for i in top_k[1:]:
        print(f"cos {float(W[i]):.3f}:{vocab.idx_to_token[i]}")

In [47]:
get_similiar_word('like',3,embeds[0],vocab)

cos 0.457:michael
cos 0.445:so
cos 0.424:capitalism
